In [ ]:
!pip install torch transformers umap-learn seaborn

In [ ]:
# Imports
from transformers import AutoModelForSequenceClassification, AutoTokenizer

import random, os, json
from itertools import product
from collections import defaultdict

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data as data

import numpy as np
import umap
import matplotlib.pyplot as plt
import seaborn as sns

MODEL_NAME = "./codebert-base-mlm"

In [ ]:
# Load dataset

f = open("./data/packet_inspection/packets_dataset.jsonl", "r")
ai_dataset = [json.loads(line) for line in f.readlines()]
f.close()

In [ ]:
# Getting Data activations

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, output_hidden_states=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
p = pipeline(model=model, tokenizer=tokenizer, return_all_scores=True)
layer = 10

inputs = [s["text"] for s in ai_dataset]
tokens_id = tokenizer(inputs, padding=True, truncation=True, return_tensors="pt")
activations =  model(**tokens_id).hidden_states


num_samples, num_tokens, _ = activations[layer].shape
tokens = [{"tokens_str": ["CLS"] + tokenizer.tokenize(s["text"], truncation=True,padding="max_length", max_length=num_tokens) + ["</s>"], "true_class": s["class"], "pred_class": p(s["text"])[0]["label"]} for s in ai_dataset]

print("Done gathering inputs")


In [ ]:
class SAE(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(SAE, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(hidden_dim, input_dim),
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded, encoded

In [ ]:
# Utility functions for SAE analysis

def get_feature_directions(W_dec):
    """
    Calcola i vettori di direzione delle feature a partire dalla matrice dei pesi del decoder.

    Args:
        W_dec (torch.Tensor): Pesi del decoder di forma (D, F), dove
                              D è la dimensione residua e F la dimensione delle feature.

    Returns:
        torch.Tensor: I vettori delle feature (direzioni normalizzate), di forma (D, F).
    """
    feature_directions = F.normalize(W_dec, p=2, dim=0, eps=1e-8)
    
    return feature_directions

def get_feature_activations(x, W_enc, b_enc, W_dec):
    """
    Calcola le attivazioni delle feature per un dato input x.

    Args:
        x (torch.Tensor): Tensore dei dati di input, di forma (batch_size, D).
        W_enc (torch.Tensor): Pesi dell'encoder, di forma (F, D).
        b_enc (torch.Tensor): Bias dell'encoder, di forma (F,).
        W_dec (torch.Tensor): Pesi del decoder, di forma (D, F).

    Returns:
        torch.Tensor: Le attivazioni delle feature, di forma (batch_size, F).
    """
    # Calcolo delle attivazioni dei feature (f_i(x) = ReLU(W_enc * x + b_enc))
    f_x = torch.relu(torch.matmul(x, W_enc.T) + b_enc)
    
    norm_W_dec = torch.linalg.norm(W_dec, dim=0)
    
    feature_activations = f_x * norm_W_dec
    
    return feature_activations

In [ ]:
MODES = ["standard", "full", "new"]
hidden_dims = [768, 768*2, 768*4, 768*8, 768*16, 768*32] 
layer = 10
beta = 5.0
lr = 5e-5
input_dim = 768
threshold = 0.01

In [ ]:
feature_to_visualize = {}

true_class_common = {}   # intersezione (feature attive in TUTTI i sample della classe)
true_class_any = {}      # unione (feature attive in ALMENO un sample della classe)

pred_class_common = {}   # intersezione (feature attive in TUTTI i sample della classe)
pred_class_any = {}      # unione (feature attive in ALMENO un sample

f_out = open("feature_stats.txt", "w")

for mode in MODES:
    feature_to_visualize[mode] = {}
    
    true_class_common[mode] = {}   # intersezione (feature attive in TUTTI i sample della classe)
    true_class_any[mode] = {}      # unione (feature attive in ALMENO un sample della classe)
    
    pred_class_common[mode] = {}   # intersezione (feature attive in TUTTI i sample della classe)
    pred_class_any[mode] = {}      # unione (feature attive in ALMENO un sample della classe)
    
    print("-"*8, mode, "-"*8)
    f_out.write("-"*8 + " " + mode + " " + "-"*8 + "\n")
    
    for hidden_dim in hidden_dims:
        feature_to_visualize[mode][hidden_dim] = set()
        
        top_features_for_true_class = {}
        top_features_for_pred_class = {}
        
        sae = SAE(input_dim, hidden_dim)
        sae.load_state_dict(torch.load(f"saved_models/sae_layer_{layer}_hiddim_{hidden_dim}_{mode}.pt"))

        encoder_weights = sae.encoder[0].weight
        decoder_weights = sae.decoder[0].weight
        feature_directions = get_feature_directions(decoder_weights)
        feature_activations = get_feature_activations(activations[layer], encoder_weights, sae.encoder[0].bias, decoder_weights) # (num_samples, num_tokens, hidden_dim)

        true_class_common[mode][hidden_dim] = {}
        true_class_any[mode][hidden_dim] = {}
        
        pred_class_common[mode][hidden_dim] = {}
        pred_class_any[mode][hidden_dim] = {}
        
        num_samples, num_tokens, hidden_dim = feature_activations.shape
        
        fa_stats = []
        
        for sample_idx in tqdm(range(num_samples)):
            
            true_class = tokens[sample_idx]["true_class"]
            pred_class = tokens[sample_idx]["pred_class"]
            
            fa_stats.append({
                "true_class": true_class,
                "pred_class": pred_class,
                "fa_avg": feature_activations[sample_idx].mean(dim=0).cpu().detach().numpy(),  # media sulle tokens
                "fa_max": feature_activations[sample_idx].max(dim=0).values.cpu().detach().numpy()  # max sulle tokens
            })
            
            active_features = set()
            mask = (feature_activations[sample_idx] > threshold).any(dim=0)  # (hidden_dim,) boolean tensor
            active_idx = torch.nonzero(mask, as_tuple=False).squeeze(-1)
            active_features = set(active_idx.tolist()) if active_idx.numel() > 0 else set()
            
            if true_class_common[mode][hidden_dim].get(true_class) is None:
                true_class_common[mode][hidden_dim][true_class] = set(active_features)
            else:
                true_class_common[mode][hidden_dim][true_class].intersection(active_features)
                
            if true_class_any[mode][hidden_dim].get(true_class) is None:
                true_class_any[mode][hidden_dim][true_class] = set(active_features)
            else:
                true_class_any[mode][hidden_dim][true_class].union(active_features)
                
            if pred_class_common[mode][hidden_dim].get(pred_class) is None:
                pred_class_common[mode][hidden_dim][pred_class] = set(active_features)
            else:
                pred_class_common[mode][hidden_dim][pred_class].intersection(active_features)
                
            if pred_class_any[mode][hidden_dim].get(pred_class) is None:
                pred_class_any[mode][hidden_dim][pred_class] = set(active_features)
            else:
                pred_class_any[mode][hidden_dim][pred_class].union(active_features)
                
        for c in true_class_common[mode][hidden_dim]:
            feature_avg_class = {}

            for f in true_class_common[mode][hidden_dim][c]:
                feature_avg = 0.0
                count_sample_feature = 0
                for s in fa_stats:
                    if s["true_class"] == c:
                        feature_avg += s["fa_avg"][f]
                        count_sample_feature += 1
                feature_avg_class[f] = feature_avg / count_sample_feature if count_sample_feature > 0 else 0.0
            
            feature_avg_class = dict(sorted(feature_avg_class.items(), key=lambda item: item[1], reverse=True))
            top_features_for_true_class[c] = dict(list(feature_avg_class.items())[:50])
            # print("top features for true class", [f for f in top_features_for_true_class[c]])

        print([c for c in pred_class_common[mode][hidden_dim]])
        for c in pred_class_common[mode][hidden_dim]:
            feature_avg_class = {}    
            for f in pred_class_common[mode][hidden_dim][c]:
                feature_avg = 0.0
                count_sample_feature = 0
                for s in fa_stats:
                    if s["pred_class"] == c:
                        feature_avg += s["fa_avg"][f]
                        count_sample_feature += 1
                feature_avg_class[f] = feature_avg / count_sample_feature if count_sample_feature > 0 else 0.0
            feature_avg_class = dict(sorted(feature_avg_class.items(), key=lambda item: item[1], reverse=True))
            top_features_for_pred_class[c] = dict(list(feature_avg_class.items())[:50])
            # print("top features for pred class", [f for f in top_features_for_pred_class[c]])
        
        common_features = set()
        for c in top_features_for_true_class:
            features = set([f for f in top_features_for_true_class[c]])
            if not common_features:
                common_features = features
            else:
                common_features = common_features & features
                # print(common_features)
                if not common_features:
                    break

        feature_to_visualize[mode][hidden_dim].update(common_features)

        # print(hidden_dim, "DEBUG: common true feature", common_features)
        for f in common_features:
            min_value = (float('inf'), "")
            max_value = (float('-inf'), "")
            avg_value = 0.0
            count = 0
            for c in top_features_for_true_class:
                avg_value += top_features_for_true_class[c][f]
                count += 1
                if top_features_for_true_class[c][f] < min_value[0]:
                    min_value = (top_features_for_true_class[c][f], c)
                if top_features_for_true_class[c][f] > max_value[0]:
                    max_value = (top_features_for_true_class[c][f], c)
            avg_value /= count
            print(f"Mode: {mode}, Hidden Dim: {hidden_dim}, True Class common feature: {f} -- Min: {min_value[0]:.6f}, {min_value[1]} | Max: {max_value[0]:.6f}, {max_value[1]} | Avg: {avg_value:.6f}")
            f_out.write(f"Mode: {mode}, Hidden Dim: {hidden_dim}, True Class common feature: {f} -- Min: {min_value[0]:.6f}, {min_value[1]} | Max: {max_value[0]:.6f}, {max_value[1]} | Avg: {avg_value:.6f}\n")

        for c in top_features_for_true_class:
            print(f"Mode: {mode}, Hidden Dim: {hidden_dim}, True Class: {c} - Top 10 features (feature: avg):")
            f_out.write(f"Mode: {mode}, Hidden Dim: {hidden_dim}, True Class: {c} - Top 10 features (feature: avg):\n")
            filtered_features = [f for f in top_features_for_true_class[c] if f not in common_features]
            feature_to_visualize[mode][hidden_dim].update(filtered_features[:10])
            for i, f in enumerate(filtered_features[:10]):
                print(f"{i+1:2d}. Feature {f}: {top_features_for_true_class[c][f]:.6f}")
                f_out.write(f"{i+1:2d}. Feature {f}: {top_features_for_true_class[c][f]:.6f}\n")

        common_features = set()
        for c in top_features_for_pred_class:
            features = set([f for f in top_features_for_pred_class[c]])
            if not common_features:
                common_features = features
            else:
                common_features = common_features & features
                # print(common_features)
                if not common_features:
                    break
        
        feature_to_visualize[mode][hidden_dim].update(common_features)
              
        # print(hidden_dim, "DEBUG: common pred feature", common_features)
        for f in common_features:
            min_value = (float('inf'), "")
            max_value = (float('-inf'), "")
            avg_value = 0.0
            count = 0
            for c in top_features_for_pred_class:
                avg_value += top_features_for_pred_class[c][f]
                count += 1
                if top_features_for_pred_class[c][f] < min_value[0]:
                    min_value = (top_features_for_pred_class[c][f], c)
                if top_features_for_pred_class[c][f] > max_value[0]:
                    max_value = (top_features_for_pred_class[c][f], c)
            avg_value /= count
            print(f"Mode: {mode}, Hidden Dim: {hidden_dim}, Pred Class common feature: {f} -- Min: {min_value[0]:.6f}, {min_value[1]} | Max: {max_value[0]:.6f}, {max_value[1]} | Avg: {avg_value:.6f}")
            f_out.write(f"Mode: {mode}, Hidden Dim: {hidden_dim}, Pred Class common feature: {f} -- Min: {min_value[0]:.6f}, {min_value[1]} | Max: {max_value[0]:.6f}, {max_value[1]} | Avg: {avg_value:.6f}\n")
            
        for c in top_features_for_pred_class:
            print(f"Mode: {mode}, Hidden Dim: {hidden_dim}, Pred Class: {c} - Top 10 features (feature: avg):")
            f_out.write(f"Mode: {mode}, Hidden Dim: {hidden_dim}, Pred Class: {c} - Top 10 features (feature: avg):\n")
            filtered_features = [f for f in top_features_for_pred_class[c] if f not in common_features]
            feature_to_visualize[mode][hidden_dim].update(filtered_features[:10])
            for i, f in enumerate(filtered_features[:10]):
                print(f"  {i+1:2d}. Feature {f}: {top_features_for_pred_class[c][f]:.6f}")
                f_out.write(f"  {i+1:2d}. Feature {f}: {top_features_for_pred_class[c][f]:.6f}\n")

                


In [ ]:
# Build and visualize class-class correlation matrices (fraction of top-50 features in common)
# per mode and hidden_dim. Assumes variables/functions from previous cells are available:
# MODES, hidden_dims, layer, activations, tokens, SAE, get_feature_activations, plt, sns, torch, np

denom_k = 50  # top-k features to consider (will adjust if hidden_dim < 50)

for mode in MODES:
    for hidden_dim in hidden_dims:
        print(f"\nMode={mode} | HiddenDim={hidden_dim}")
        # load SAE
        sae = SAE(input_dim, hidden_dim)
        sae.load_state_dict(torch.load(f"saved_models/sae_layer_{layer}_hiddim_{hidden_dim}_{mode}.pt"))
        encoder_weights = sae.encoder[0].weight      # shape (F, D)
        decoder_weights = sae.decoder[0].weight      # shape (D, F)
        b_enc = sae.encoder[0].bias

        # compute feature activations: shape (num_samples, num_tokens, F)
        fa = get_feature_activations(activations[layer], encoder_weights, b_enc, decoder_weights)
        # average over tokens -> (num_samples, F)
        fa_avg_per_sample = fa.mean(dim=1).cpu().detach().numpy()

        # group samples by true class and compute average activation per feature for each class
        class_to_indices = {}
        for i, t in enumerate(tokens):
            c = t["true_class"]
            class_to_indices.setdefault(c, []).append(i)

        class_to_topk = {}
        actual_k = min(denom_k, hidden_dim)
        for c, idxs in class_to_indices.items():
            # mean across samples of this class
            class_mean = fa_avg_per_sample[idxs].mean(axis=0) if len(idxs) > 0 else np.zeros(hidden_dim)
            topk_idx = np.argsort(class_mean)[::-1][:actual_k]
            class_to_topk[c] = set(int(x) for x in topk_idx)

        classes = sorted(class_to_topk.keys())
        n = len(classes)
        if n == 0:
            print("  No classes found, skipping.")
            continue

        # build correlation matrix: fraction of shared features in top-K
        mat = np.zeros((n, n), dtype=float)
        for i, ci in enumerate(classes):
            for j, cj in enumerate(classes):
                inter = len(class_to_topk[ci].intersection(class_to_topk[cj]))
                mat[i, j] = inter / actual_k

        # print a small summary table
        print("  Classes:", classes)
        print("  Correlation matrix (rows/cols = classes, values = shared_topk / topk):")
        np.set_printoptions(precision=3, suppress=True)
        print(mat)

        # plot heatmap
        plt.figure(figsize=(max(6, n*0.6), max(5, n*0.5)))
        sns.heatmap(mat, xticklabels=classes, yticklabels=classes, annot=True, fmt=".2f", cmap="viridis")
        plt.title(f"Mode={mode} | HiddenDim={hidden_dim} | top-{actual_k} overlap fraction")
        plt.tight_layout()
        plt.show()

In [ ]:
for mode in MODES:
    for hidden_dim in hidden_dims:
        top_features_for_true_class = {}
        top_features_for_pred_class = {}
        
        sae = SAE(input_dim, hidden_dim)
        sae.load_state_dict(torch.load(f"saved_models/sae_layer_{layer}_hiddim_{hidden_dim}_{mode}.pt"))

        encoder_weights = sae.encoder[0].weight
        decoder_weights = sae.decoder[0].weight
        feature_directions = get_feature_directions(decoder_weights)
        feature_activations = get_feature_activations(activations[layer], encoder_weights, sae.encoder[0].bias, decoder_weights) # (num_samples, num_tokens, hidden_dim)

        num_samples, num_tokens, hidden_dim = feature_activations.shape
        
        out_path = f"feature_visualizations_hiddim_{hidden_dim}_{mode}.json"
        with f as open(out_path, "w"):
            output = []
            for sample_idx in range(num_samples):
                
                true_class = tokens[sample_idx]["true_class"]
                pred_class = tokens[sample_idx]["pred_class"]
                tokens_json = []
                for tok_idx in range(num_tokens):
                    token_str = tokens[sample_idx]["tokens_str"][tok_idx] if tok_idx < len(tokens[sample_idx]["tokens_str"]) else "[PAD]"
                    if token_str != "[PAD]":
                        for f in feature_to_visualize[mode][hidden_dim]:
                            vec = feature_activations[sample_idx, tok_idx].cpu()
                            nonzero_idx = torch.nonzero(vec != 0, as_tuple=False).squeeze(-1)
                            activations_ = []
                            if nonzero_idx.numel() > 0:
                                # filtra solo le feature presenti in allowed_features
                                for fid in nonzero_idx.tolist():
                                    if fid in allowed_features:
                                        activations_.append((int(fid), float(vec[fid].item())))

                            tokens_json.append({
                                "token_idx": tok_idx,
                                "token_str": token_str,
                                "activations": activations_
                            })
                    else:
                        break

                output.append({
                    "sample_index": sample_idx,
                    "true_class": true_class,
                    "pred_class": pred_class,
                    "tokens": tokens_json
                })
            f.write(json.dumps(output) + "\n")
            print(f"Wrote feature visualization data to {out_path}")